# PyTorch Oefening: Een Simpel Neuraal Netwerk Trainen

In deze oefening maken we kennis met PyTorch en trainen we een simpel neuraal netwerk op de image-classification "MNIST" dataset (handgeschreven cijfers 0-9).

We gaan niet uitgebreid in op de concepten die hier aan bod komen, maar als je het interessant vindt, waar het eigenlijk allemaal mee begon, lees dan over de Feedforward Neural Network hier: https://medium.com/hackernoon/deep-learning-feedforward-neural-networks-explained-c34ae3f084f1

**Doel:** classificeren van het getal in de afbeelding (0-9)


In [ ]:
# Dit vereist de GCP instance, maar is al geïnstalleerd daar.
!pip install numpy
!pip install torchinfo
!pip install matplotlib
!pip install torch
!pip install torchvision

## 1. Imports

**Voordeel van voor-geïnstalleerde kernels**:

Cloud omgevingen bieden kernels met PyTorch pre-installed, wat zorgt voor een snelle opstart zonder installatietijd. Bij gebruik van GPUs (niet in deze workshop), zijn de GPU-drivers meteen correct geconfigureerd.

- **torch**: Core library voor deep learning
- **torchvision**: Computer vision utilities met pre-trained modellen, datasets en image transformaties

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## 2. Data Laden

We gebruiken de MNIST dataset: 28x28 grijswaarde afbeeldingen van handgeschreven cijfers (0-9)


In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

**Opdracht:**
- Print hoeveel training en test samples je dataset heeft

In [ ]:
# Implementeer hier

**Opdracht:** Run de volgende cel, analyseer de code en bekijk de data samples

In [ ]:
import matplotlib.pyplot as plt

# Laat een paar voorbeelden zien
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    image, label = train_data[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.tight_layout()
plt.savefig('mnist_voorbeelden.png', dpi=300, bbox_inches='tight')


## 3. Model Definiëren

We gebruiken een blauwdruk, een zogeten **class**, voor het maken van objecten met bepaalde eigenschappen en gedrag. Je kunt ervoor zorgen dat je object de eigenschappen erft van een bestaande class. In dit geval: `nn.Module`.

We maken een simpel **feedforward neuraal netwerk** (FNN).
> Een **FNN** is een neuraal netwerk waarbij informatie maar één kant op stroomt: van input naar output, zonder terugkoppellussen. Het bestaat uit lagen neuronen die telkens een gewogen som en een activatiefunctie toepassen. Het model leert door de gewichten aan te passen zodat de output beter past bij de gewenste antwoorden.

**Opdracht:**
Als je de cell uitvoert zie je `in_features=784`, dit staat voor *input* features. Achterhaal waarom dit 784 is, en bedenk wat deze input features voorstellen.


In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 128)  # Eerste laag: 784 → 128
        self.relu = nn.ReLU()             # Activatiefunctie
        self.fc2 = nn.Linear(128, 10)     # Output laag: 128 → 10
    
    def forward(self, x):
        x = self.flatten(x)     # Maak 2D image plat naar 1D vector
        x = self.relu(self.fc1(x))
        return self.fc2(x)


model = SimpleNN()
print(model)
print("Mooi! Je hebt nu een neuraal netwerk gedefinieerd in Python")

In [ ]:
# Beantwoord hier waarom in_features gelijk is aan 784.

**Opdracht**:

Bereken hoeveel trainbare parameters het model heeft. Gebruik summary(model) of tel handmatig.

In [ ]:
# Print hier hoeveel parameters het model heeft, met summary(model)


## 4. Loss Functie, Optimizer en Learning rate

Een **Loss Functie** meet hoe ver de voorspellingen van het model af liggen van de werkelijke waarden met. We gebruiken `CrossEntropyLoss` voor classificatie.

De **optimizer** past de gewichten van het netwerk aan om de loss te minimaliseren. We gebruiken `Adam` met een **learning rate** (lr) van 0.001. De lr bepaalt hoe groot de stappen zijn die het model neemt tijdens het leren. Te groot = instabiel, te klein = traag leren.

In [ ]:
criterion = nn.CrossEntropyLoss()  # Loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)

## 5. Evalueer je model
Vòòr en na het trainen kun je kijken wat de prestatie van je model is op een test set.

**Opdracht:**
Bereken de nauwkeurigheid (of *accuracy*) van het model door een simpele berekening te doen met `correct` en `total`


In [ ]:
# Test data laden
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

# Evalueer model
model.eval()  # Zet model in evaluatie modus
correct = 0
total = 0

with torch.no_grad():  # Geen gradiënten nodig tijdens evaluatie
    for data, target in test_loader:
        output = model(data)
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

# Implementeer hier de accuracy en print deze uit

## 5. Model Trainen

### Epochs
Een **epoch** is één volledige doorloop door alle training data. We trainen 2 epochs.

### Training Loop
Voor elke batch data:
1. `optimizer.zero_grad()` - Reset gradiënten
2. `output = model(data)` - Voorspelling maken
3. `loss = criterion(output, target)` - Bereken loss
4. `loss.backward()` - Bereken gradiënten (backpropagation)
5. `optimizer.step()` - Update gewichten

In [ ]:
epochs = 2

for epoch in range(epochs):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()       # Reset gradiënten
        output = model(data)        # Voorspelling
        loss = criterion(output, target)  # Bereken loss
        loss.backward()             # Backpropagation
        optimizer.step()            # Update gewichten
        
        if batch_idx % 100 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Batch {batch_idx}, Loss: {loss.item():.4f}')

print("\nTraining voltooid!")

**Opdracht:** 
Meet de nieuwe prestatie van je model met de code in sectie 5. 

Is de score verbeterd?

In [ ]:
# Beantwoord hier de opdracht

**Opdracht:**
Wijzig de learning rate en observeer het effect

Experimenteer met verschillende learning rates (cell ergens hierboven)
- `lr=0.01` (10x hoger)
- `lr=0.0001` (10x lager)

(!) Let op: run opnieuw de cellen vanaf sectie 3, zodat je getrainde model eerst wordt vergeten.

Wat gebeurt er met:
- De snelheid waarmee de loss daalt?
- De stabiliteit van de training?


In [ ]:
# Beantwoord hier de opdracht

### Bonus 1: Voeg een extra hidden layer toe

Maak een dieper netwerk met 2 hidden layers:
- Input → 128 → 64 → Output

Vergeet de activatiefunctie (ReLU) tussen de layers niet!

In [ ]:
# Implementeer hier

### Bonus 2: Experimenteer met het aantal neuronen in de hidden layer

Wijzig het aantal neuronen in de hidden layer van 128 naar 256. Train het model opnieuw en observeer:
- Verandert het aantal parameters?
- Wordt de loss lager?
- Duurt training langer?

In [ ]:
# Beantwoord hier de opdracht